# Canine Spleen Benchmark

Import selected canine CT series from Orthanc, run MONAI Label's human `segmentation_spleen` model zero-shot, and compare predictions with same-basename radiologist NIfTI masks.

**Research use only.** The pretrained model was developed on human CT and is not validated for canine clinical use. Keep credentials and all image-derived data outside Git.

## 1. Preflight

Create `.env` from `.env.example`, start MONAI Label with `docker compose up -d monailabel`, and put reference masks in `data/radiologist_masks/<case_id>.nii.gz`. Set `RUN_LIVE = True` only when the local environment is ready.

In [ ]:
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
from IPython.display import display

from spleen_segmenter.config import Settings
from spleen_segmenter.evaluation import evaluate_case, geometry, geometry_matches, pair_cases
from spleen_segmenter.monailabel import MonaiLabelClient
from spleen_segmenter.orthanc import OrthancClient, import_series

ROOT = Path.cwd().resolve()
RUN_LIVE = False
MODEL_NAME = "segmentation_spleen"
ALLOW_PREDICTION_RESAMPLE = False
print(f"Project root: {ROOT}")

In [ ]:
settings = None
if RUN_LIVE:
    settings = Settings.from_env(ROOT)
    settings.ensure_local_directories()
    display(settings.public_summary())  # Credentials are intentionally omitted.
    if shutil.which("dcm2niix") is None:
        raise RuntimeError("dcm2niix is required for Orthanc DICOM conversion")
else:
    print("Dry mode: network, import, and inference cells will be skipped.")

## 2. Discover CT series in Orthanc

`ORTHANC_QUERY` is sent to `/tools/find` together with `Modality=CT`. Adjust it for tags supported by your archive. Returned notebook data is restricted to Orthanc series ID, modality, description, number, and instance count; patient tags are not returned.

In [ ]:
ORTHANC_QUERY = {"SpeciesDescription": "CANINE"}
SEARCH_LIMIT = 100

orthanc = None
series = []
if RUN_LIVE:
    orthanc = OrthancClient(
        settings.orthanc_url,
        username=settings.orthanc_username,
        password=settings.orthanc_password,
        verify_tls=settings.orthanc_verify_tls,
        timeout_seconds=settings.orthanc_timeout_seconds,
    )
    display(orthanc.system_info())
    series = orthanc.find_ct_series(ORTHANC_QUERY, limit=SEARCH_LIMIT)
    display(pd.DataFrame([{
        "orthanc_series_id": item.orthanc_series_id,
        "modality": item.modality,
        "series_description": item.series_description,
        "series_number": item.series_number,
        "instance_count": item.instance_count,
    } for item in series]))

## 3. Select and import series

Map each Orthanc series ID to the exact case basename used by its radiologist mask. Leaving the dictionary empty imports nothing. Temporary archives and DICOM files are deleted after conversion; the local ignored manifest stores only case ID, Orthanc series ID, and CT path.

In [ ]:
SELECTED_SERIES = {
    # "orthanc-series-id": "matching-radiologist-mask-basename",
}

imported = []
import_errors = []
if RUN_LIVE:
    for series_id, case_id in SELECTED_SERIES.items():
        try:
            imported.append(import_series(
                orthanc,
                series_id,
                settings.ct_nifti_dir,
                case_id=case_id,
                manifest_path=ROOT / "data" / "import_manifest.csv",
            ))
        except Exception as exc:
            import_errors.append({"case_id": case_id, "error": str(exc)})
    display(pd.DataFrame([{"case_id": r.case_id, "ct_path": r.ct_path} for r in imported]))
    if import_errors:
        display(pd.DataFrame(import_errors))

## 4. Pair CTs with radiologist masks and validate geometry

CT and reference mask must have the same 3D shape and affine before inference/evaluation. Resolve registration or export errors at the source rather than silently resampling ground truth.

In [ ]:
if RUN_LIVE:
    pairing = pair_cases(settings.ct_nifti_dir, settings.radiologist_mask_dir)
    print(f"Paired cases: {len(pairing.cases)}")
    print(f"CTs missing references: {pairing.missing_references}")
    print(f"Orphan references: {pairing.orphan_references}")
    geometry_rows = []
    valid_cases = []
    for case in pairing.cases:
        ct_image = nib.load(case.ct_path)
        reference_image = nib.load(case.reference_path)
        matches = geometry_matches(ct_image, reference_image)
        geometry_rows.append({
            "case_id": case.case_id,
            "geometry_matches": matches,
            "ct": geometry(ct_image),
            "reference": geometry(reference_image),
        })
        if matches:
            valid_cases.append(case)
    display(pd.DataFrame(geometry_rows))
else:
    pairing = None
    valid_cases = []

## 5. Run zero-shot MONAI Label inference

The client verifies that `segmentation_spleen` is loaded, uploads each geometry-valid CT, and atomically saves the returned NIfTI mask. Existing predictions are not overwritten.

In [ ]:
inference_rows = []
if RUN_LIVE:
    monai = MonaiLabelClient(
        settings.monai_label_url,
        timeout_seconds=settings.monai_label_timeout_seconds,
    )
    model_info = monai.require_model(MODEL_NAME)
    print(f"MONAI Label is ready; model={MODEL_NAME}")
    for case in valid_cases:
        prediction_path = settings.prediction_dir / f"{case.case_id}.nii.gz"
        try:
            if prediction_path.exists():
                status = "existing"
            else:
                monai.infer(case.ct_path, prediction_path, model=MODEL_NAME)
                status = "created"
            inference_rows.append({"case_id": case.case_id, "status": status, "error": ""})
        except Exception as exc:
            inference_rows.append({"case_id": case.case_id, "status": "failed", "error": str(exc)})
    display(pd.DataFrame(inference_rows))

## 6. Evaluate overlap, surfaces, and volumes

Prediction resampling is off by default. If enabled after review, only predictions are resampled to reference geometry with nearest-neighbor interpolation, and the adjustment is recorded per case.

In [ ]:
metric_rows = []
evaluation_errors = []
if RUN_LIVE:
    scored = pair_cases(
        settings.ct_nifti_dir,
        settings.radiologist_mask_dir,
        settings.prediction_dir,
    )
    for case in scored.cases:
        if case.prediction_path is None:
            evaluation_errors.append({"case_id": case.case_id, "error": "prediction missing"})
            continue
        try:
            metrics, reference_geometry, prediction_geometry = evaluate_case(
                case.case_id,
                case.reference_path,
                case.prediction_path,
                allow_resample=ALLOW_PREDICTION_RESAMPLE,
            )
            metric_rows.append(metrics.to_dict())
        except Exception as exc:
            evaluation_errors.append({"case_id": case.case_id, "error": str(exc)})

metrics_df = pd.DataFrame(metric_rows)
display(metrics_df)
if not metrics_df.empty:
    display(metrics_df.describe(include="all"))
    settings.report_dir.mkdir(parents=True, exist_ok=True)
    metrics_df.to_csv(settings.report_dir / "case_metrics.csv", index=False)
if evaluation_errors:
    display(pd.DataFrame(evaluation_errors))

## 7. Review overlays

Visual review is essential. The plot shows the axial slice with the largest radiologist mask area: radiologist contour in green and MONAI contour in red.

In [ ]:
def show_overlay(case):
    ct = np.asanyarray(nib.load(case.ct_path).dataobj)
    reference = np.asanyarray(nib.load(case.reference_path).dataobj) > 0
    prediction = np.asanyarray(nib.load(case.prediction_path).dataobj) > 0
    if ct.shape != reference.shape or prediction.shape != reference.shape:
        raise ValueError("Overlay requires matching voxel arrays; review geometry first")
    axial_index = int(np.argmax(reference.sum(axis=(0, 1))))
    image_slice = ct[:, :, axial_index].T
    ref_slice = reference[:, :, axial_index].T
    pred_slice = prediction[:, :, axial_index].T
    lower, upper = np.percentile(image_slice[np.isfinite(image_slice)], [1, 99])
    fig, axis = plt.subplots(figsize=(7, 7))
    axis.imshow(image_slice, cmap="gray", origin="lower", vmin=lower, vmax=upper)
    if ref_slice.any():
        axis.contour(ref_slice, levels=[0.5], colors=["lime"], linewidths=1.5)
    if pred_slice.any():
        axis.contour(pred_slice, levels=[0.5], colors=["red"], linewidths=1.5)
    axis.set_title(f"{case.case_id} — axial {axial_index} (reference=green, MONAI=red)")
    axis.axis("off")
    plt.show()

if RUN_LIVE and 'scored' in globals():
    overlay_cases = [case for case in scored.cases if case.prediction_path is not None]
    if overlay_cases:
        show_overlay(overlay_cases[0])

## Interpretation checklist

- Treat this as external-domain, zero-shot performance of a human-CT model.
- Report scanner/protocol and contrast-phase strata where sample size permits.
- Investigate every geometry failure and visually review every prediction.
- Do not tune thresholds or exclude failures based on test-set radiologist masks.
- Keep de-identification and study approvals outside this public repository.